In [1]:
# Memory-Optimized HDF5 Dataset Preparation (Streaming Version)
# Processes samples one-by-one with aggressive memory management
# Designed to handle very large files without crashing


In [1]:
from pathlib import Path
import numpy as np
import h5py
import json
import logging
from tqdm import tqdm
from typing import Dict, List, Optional, Tuple, Generator
from sklearn.model_selection import train_test_split
import warnings
import gc
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

AUGMENTED_DATA_DIR = Path("/Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/testing")
OUTPUT_BASE_DIR   = Path("/Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/hdf5_datasets")

# --- Configuration ---
REQUIRE_ALL_KEYS = False    # Allow M11s fallback
MUELLER_INPUT_KEY = 'nM'
NM11S_KEY         = 'nM11s'
M11S_KEY          = 'M11s'
MASK_KEYS         = ('tissue_mask', 'annotation_mask')

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Memory optimization settings
VALIDATION_BATCH_SIZE = 100  # Process validation in batches
STATISTICS_SAMPLE_SIZE = 10  # Number of samples for statistics
COMPRESSION_LEVEL = 4        # Lower = faster, less memory, more disk space
ENABLE_COMPRESSION = True    # Set to False if you have disk space

print("=" * 80)
print("MEMORY-OPTIMIZED HDF5 DATASET PREPARATION (STREAMING)")
print("=" * 80)
print(f"Source: {AUGMENTED_DATA_DIR}")
print(f"Output: {OUTPUT_BASE_DIR}")
print(f"Compression: {'Enabled (level ' + str(COMPRESSION_LEVEL) + ')' if ENABLE_COMPRESSION else 'Disabled'}")


MEMORY-OPTIMIZED HDF5 DATASET PREPARATION (STREAMING)
Source: /Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/testing
Output: /Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/hdf5_datasets
Compression: Enabled (level 4)


In [2]:
def find_mask_key(d: Dict) -> Optional[str]:
    """Find mask key in dictionary"""
    for k in MASK_KEYS:
        if k in d:
            try:
                val = d[k]
                if isinstance(val, np.ndarray):
                    if val.size > 0:
                        return k
            except:
                continue
    return None

def reshape_nM_to_16(nM: np.ndarray) -> Optional[np.ndarray]:
    if nM.ndim == 4 and nM.shape[2:] == (4,4):
        h, w = nM.shape[:2]
        return nM.reshape(h, w, 16)
    if nM.ndim == 3 and nM.shape[2] == 16:
        return nM
    return None

def to_float32(x: np.ndarray) -> np.ndarray:
    return x.astype(np.float32, copy=False)

def to_mask_bool(x: np.ndarray) -> np.ndarray:
    return x if x.dtype == np.bool_ else (x != 0)

def to_mask_int(x: np.ndarray) -> np.ndarray:
    return x.astype(np.int64, copy=False)

def normalize_to_01(x: np.ndarray) -> Tuple[np.ndarray, float, float]:
    x_min = float(np.min(x))
    x_max = float(np.max(x))
    if x_max > x_min:
        y = (x - x_min) / (x_max - x_min)
    else:
        y = np.zeros_like(x, dtype=np.float32)
    return y.astype(np.float32, copy=False), x_min, x_max


In [3]:
class StreamingHDF5Creator:
    """Memory-efficient HDF5 creator that streams data directly to disk"""
    
    def __init__(self, augmented_dir: Path, output_dir: Path):
        self.src = Path(augmented_dir)
        self.out = Path(output_dir) / 'hdf5' / 'streaming'
        self.out.mkdir(parents=True, exist_ok=True)
        logger.info(f"Initialized: {self.src} -> {self.out}")

    def discover(self) -> List[Path]:
        """Discover all NPZ files, filtering hidden files"""
        if not self.src.exists():
            logger.error(f"Missing source directory: {self.src}")
            return []
        
        all_files = list(self.src.glob("*/*.npz"))
        files = [f for f in all_files if not f.name.startswith('._')]
        
        modalities = set([f.stem for f in files])
        logger.info(f"Found {len(files)} NPZ files (filtered {len(all_files) - len(files)} hidden files)")
        logger.info(f"Modality types: {sorted(modalities)}")
        
        return files

    def validate_sample(self, fp: Path) -> Optional[Dict]:
        """
        Validate sample and return metadata only (not the full data).
        Returns minimal dict with file info and validation status.
        """
        sample_name = fp.parent.name
        modality = fp.stem
        
        try:
            # Use mmap_mode='r' to avoid loading entire file into memory
            with np.load(fp, allow_pickle=True, mmap_mode='r') as npz_data:
                d = dict(npz_data)
                
                # Check keys
                mk = find_mask_key(d)
                has_mask   = mk is not None
                has_nM     = MUELLER_INPUT_KEY in d
                has_nM11s  = NM11S_KEY in d
                has_M11s   = M11S_KEY in d
        except Exception as e:
            logger.error(f"{sample_name}/{modality}: load error: {e}")
            return None
    
        if REQUIRE_ALL_KEYS:
            if not has_mask or not has_nM or not has_nM11s:
                return None
        else:
            if not has_mask or not has_nM or not (has_nM11s or has_M11s):
                return None
        
        # Return metadata only
        return {
            'file_path': fp,
            'sample_name': sample_name,
            'modality': modality,
            'mask_key': mk,
            'has_nM11s': has_nM11s
        }

    def load_and_process_sample(self, metadata: Dict) -> Optional[Dict]:
        """Load and process a single sample (called when writing to HDF5)"""
        fp = metadata['file_path']
        sample_name = metadata['sample_name']
        modality = metadata['modality']
        mk = metadata['mask_key']
        
        try:
            npz_data = np.load(fp, allow_pickle=True)
            d = dict(npz_data)
            npz_data.close()
        except Exception as e:
            logger.error(f"{sample_name}/{modality}: load error: {e}")
            return None
        
        # Extract mask
        try:
            mask_data = d[mk]
            if isinstance(mask_data, np.ndarray):
                if mask_data.dtype == object and mask_data.size == 1:
                    mask = np.asarray(mask_data.item())
                else:
                    mask = mask_data
            else:
                mask = np.asarray(mask_data)
            
            mask = to_mask_bool(mask)
            if mask.ndim != 2:
                return None
            target = to_mask_int(mask)
        except Exception as e:
            return None
    
        # Extract nM
        try:
            nM_raw = d[MUELLER_INPUT_KEY]
            nM_16 = reshape_nM_to_16(nM_raw)
            if nM_16 is None or nM_16.shape[:2] != target.shape:
                return None
            nM_16 = to_float32(nM_16)
        except Exception as e:
            return None
    
        # Extract nM11s or normalize M11s
        nm11_source = None
        nm11_min = None
        nm11_max = None
    
        try:
            if metadata['has_nM11s']:
                nm11 = to_float32(d[NM11S_KEY])
                if nm11.ndim != 2 or nm11.shape != target.shape:
                    return None
                nm11_source = 'nM11s'
            else:
                m11 = to_float32(d[M11S_KEY])
                if m11.ndim != 2 or m11.shape != target.shape:
                    return None
                nm11, nm11_min, nm11_max = normalize_to_01(m11)
                nm11_source = 'M11s_normalized'
        except Exception as e:
            return None
    
        # Combine - OPTIMIZED: No unnecessary copy
        X = nM_16  # Use the array directly instead of copying
        X[:,:,0] = nm11
    
        return {
            'name': f"{sample_name}_{modality}",
            'sample_name': sample_name,
            'modality': modality,
            'input': X,
            'nm11': nm11,
            'nm11_source': nm11_source,
            'nm11_min': nm11_min,
            'nm11_max': nm11_max,
            'target': target
        }

    def collect_statistics_chunked(self, file_metadatas: List[Dict], 
                                   sample_size: int = 10) -> Tuple:
        """
        Collect statistics using Welford's online algorithm for constant memory.
        Processes data in chunks to handle very large arrays.
        """
        logger.info(f"Pass 1/2: Collecting statistics from {sample_size} samples...")
        all_classes = set()
        
        # Welford's online algorithm for mean/std (constant memory)
        n = 0
        mean = 0.0
        M2 = 0.0
        min_val = float('inf')
        max_val = float('-inf')
        
        for metadata in tqdm(file_metadatas[:sample_size], desc="Sampling"):
            sample = self.load_and_process_sample(metadata)
            if not sample:
                continue
                
            X = sample['input']
            y = sample['target']
            
            # Update min/max
            min_val = min(min_val, float(X.min()))
            max_val = max(max_val, float(X.max()))
            all_classes.update(np.unique(y).tolist())
            
            # Welford's algorithm - process in chunks to avoid memory issues
            chunk_size = 10000
            X_flat = X.ravel()
            for i in range(0, len(X_flat), chunk_size):
                chunk = X_flat[i:i+chunk_size]
                for x in chunk:
                    n += 1
                    delta = x - mean
                    mean += delta / n
                    M2 += delta * (x - mean)
            
            # Free memory immediately
            del sample, X, y, X_flat
            gc.collect()
        
        std = np.sqrt(M2 / n) if n > 1 else 1.0
        
        return mean, std, min_val, max_val, all_classes

    def write_hdf5_streaming(self, file_metadatas: List[Dict], split: str, out_path: Path):
        """
        Write HDF5 file by streaming samples one at a time.
        Only keeps one sample in memory at a time.
        """
        if not file_metadatas:
            logger.warning(f"No samples for split {split}")
            return None

        logger.info(f"Creating streaming/{split}: {len(file_metadatas)} files")
        
        # Collect statistics using optimized chunked method
        mean, std, min_val, max_val, all_classes = self.collect_statistics_chunked(
            file_metadatas, 
            sample_size=min(STATISTICS_SAMPLE_SIZE, len(file_metadatas))
        )
        
        # Force garbage collection before main write
        gc.collect()
        
        # Second pass: write data
        logger.info(f"Pass 2/2: Writing data to HDF5...")
        
        # Setup compression parameters
        if ENABLE_COMPRESSION:
            compression_kwargs = {
                'compression': 'gzip',
                'compression_opts': COMPRESSION_LEVEL
            }
        else:
            compression_kwargs = {'compression': None}
        
        with h5py.File(out_path, 'w') as h5f:
            grp = h5f.create_group('samples')
            names = []
            sample_idx = 0
            
            for i, metadata in enumerate(tqdm(file_metadatas, desc=f"Writing {split}")):
                sample = self.load_and_process_sample(metadata)
                if not sample:
                    continue
                
                g = grp.create_group(f"sample_{sample_idx:06d}")
                x = sample['input'].astype(np.float32, copy=False)
                y = sample['target'].astype(np.int64, copy=False)
                nm11 = sample['nm11'].astype(np.float32, copy=False)

                # Write with optimized compression settings
                g.create_dataset('input', data=x, **compression_kwargs)
                g.create_dataset('target', data=y, **compression_kwargs)
                g.create_dataset('nM11s', data=nm11, **compression_kwargs)

                # Metadata
                g.attrs['sample_name'] = sample['name']
                g.attrs['original_sample_name'] = sample['sample_name']
                g.attrs['modality'] = sample['modality']
                g.attrs['input_shape'] = x.shape
                g.attrs['target_shape'] = y.shape
                g.attrs['file_path'] = str(metadata['file_path'])
                g.attrs['nM11s_source'] = sample['nm11_source']
                if sample['nm11_source'] == 'M11s_normalized':
                    g.attrs['M11s_min'] = float(sample['nm11_min'])
                    g.attrs['M11s_max'] = float(sample['nm11_max'])

                names.append(sample['name'])
                sample_idx += 1
                
                # Aggressive memory cleanup
                del sample, x, y, nm11
                
                # Force garbage collection every 50 samples
                if (i + 1) % 50 == 0:
                    gc.collect()

            # Dataset-level metadata
            h5f.create_dataset('sample_names',
                               data=[n.encode() for n in names],
                               dtype=h5py.string_dtype())

            # File attributes
            h5f.attrs['dataset_type'] = 'streaming_combined_nM_with_c0_as_nM11s'
            h5f.attrs['split'] = split
            h5f.attrs['num_samples'] = sample_idx
            h5f.attrs['target_type'] = 'segmentation_mask'
            h5f.attrs['variable_shapes'] = True
            h5f.attrs['created_date'] = str(np.datetime64('now'))
            h5f.attrs['num_input_channels'] = 16
            h5f.attrs['channel0_source_rule'] = 'if nM11s present use it; else normalize M11s to [0,1]'
            h5f.attrs['input_channel_semantics'] = json.dumps({
                "c0": "nM11s (normalized M11 if necessary)",
                "c1..c15": "remaining flattened Mueller elements from nM"
            })

            cls = sorted(list(all_classes))
            h5f.attrs['num_classes'] = len(cls)
            h5f.create_dataset('class_labels', data=np.array(cls, dtype=np.int64))

            h5f.attrs['input_mean'] = float(mean)
            h5f.attrs['input_std'] = float(std)
            h5f.attrs['input_min'] = float(min_val)
            h5f.attrs['input_max'] = float(max_val)
            h5f.attrs['compression_enabled'] = ENABLE_COMPRESSION
            if ENABLE_COMPRESSION:
                h5f.attrs['compression_level'] = COMPRESSION_LEVEL

        logger.info(f"Created {out_path} with {sample_idx} samples")
        
        # Final cleanup
        gc.collect()
        
        return out_path

    def validate_files_batched(self, files: List[Path]) -> List[Dict]:
        """
        Validate files in batches to avoid memory buildup.
        """
        logger.info("Validating files in batches...")
        valid_metadatas = []
        
        for i in range(0, len(files), VALIDATION_BATCH_SIZE):
            batch = files[i:i+VALIDATION_BATCH_SIZE]
            batch_num = i // VALIDATION_BATCH_SIZE + 1
            total_batches = (len(files) + VALIDATION_BATCH_SIZE - 1) // VALIDATION_BATCH_SIZE
            
            for fp in tqdm(batch, desc=f"Validating batch {batch_num}/{total_batches}"):
                metadata = self.validate_sample(fp)
                if metadata:
                    valid_metadatas.append(metadata)
            
            # Cleanup after each batch
            gc.collect()
        
        return valid_metadatas

    def build(self):
        """Build HDF5 datasets using streaming approach with batched validation"""
        files = self.discover()
        if not files:
            return {}

        # Validate all files in batches (lightweight, just metadata)
        valid_metadatas = self.validate_files_batched(files)

        if not valid_metadatas:
            logger.error("No valid samples")
            return {}

        logger.info(f"Successfully validated {len(valid_metadatas)} samples")

        # Split into train/val/test
        train, temp = train_test_split(valid_metadatas, test_size=(VAL_RATIO + TEST_RATIO), random_state=42)
        val, test   = train_test_split(temp, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=42)

        # Write HDF5 files (streaming)
        results = {}
        for split_name, subset in (('train', train), ('validation', val), ('test', test)):
            if subset:
                out_path = self.out / f"{split_name}.h5"
                created = self.write_hdf5_streaming(subset, split_name, out_path)
                if created:
                    results[split_name] = {'path': str(created), 'num_samples': len(subset)}
                
                # Cleanup between splits
                gc.collect()
        
        return results

    def print_summary(self, results: Dict):
        print("\n" + "="*80)
        print("HDF5 DATASET CREATION SUMMARY — STREAMING (OPTIMIZED)")
        print("="*80)
        total = 0
        for split, info in results.items():
            n = info['num_samples']
            total += n
            file_path = Path(info['path'])
            file_size = file_path.stat().st_size / (1024**3) if file_path.exists() else 0
            print(f"  {split:>10}: {n:>6,}  {file_path.name} ({file_size:.2f} GB)")
        print(f"  {'Total':>10}: {total:>6,}")
        print(f"Output dir: {self.out}")
        return results


In [4]:
def verify_hdf5_files(hdf5_dir: Path):
    print("\n" + "="*80)
    print("VERIFY HDF5 — STREAMING (OPTIMIZED)")
    print("="*80)
    streaming_dir = hdf5_dir / 'streaming'
    if not streaming_dir.exists():
        print(f"Directory not found: {streaming_dir}")
        return
    
    for h5_file in streaming_dir.glob("*.h5"):
        print(f"\n{h5_file.name}")
        try:
            with h5py.File(h5_file, 'r') as f:
                print(f"  Samples: {f.attrs['num_samples']}")
                print(f"  Split:   {f.attrs['split']}")
                print(f"  Chans:   {f.attrs['num_input_channels']}")
                print(f"  Input μ/σ: {f.attrs['input_mean']:.4f}/{f.attrs['input_std']:.4f}")
                
                if 'compression_enabled' in f.attrs:
                    comp = f.attrs['compression_enabled']
                    if comp and 'compression_level' in f.attrs:
                        print(f"  Compression: gzip level {f.attrs['compression_level']}")
                    else:
                        print(f"  Compression: disabled")
                
                keys = list(f['samples'].keys())
                if keys:
                    g = f['samples'][keys[0]]
                    print(f"  Example: input {g['input'].shape}, target {g['target'].shape}")
                    print(f"  Sample: {g.attrs['sample_name']}")
                    print(f"  Modality: {g.attrs['modality']}")
                
                # Show file size
                file_size = h5_file.stat().st_size / (1024**3)
                print(f"  File size: {file_size:.2f} GB")
        except Exception as e:
            print(f"  ERROR: {e}")


In [5]:
if __name__ == "__main__":
    import psutil
    import os
    
    # Show memory info
    process = psutil.Process(os.getpid())
    mem_info = psutil.virtual_memory()
    logger.info(f"Available RAM: {mem_info.available / (1024**3):.1f} GB / {mem_info.total / (1024**3):.1f} GB")
    
    if not AUGMENTED_DATA_DIR.exists():
        logger.error(f"Missing source directory: {AUGMENTED_DATA_DIR}")
    else:
        creator = StreamingHDF5Creator(AUGMENTED_DATA_DIR, OUTPUT_BASE_DIR)
        res = creator.build()
        creator.print_summary(res)
        verify_hdf5_files(OUTPUT_BASE_DIR / 'hdf5')
        
        # Show final memory usage
        mem_mb = process.memory_info().rss / (1024**2)
        mem_info_final = psutil.virtual_memory()
        logger.info(f"Peak memory usage: {mem_mb:.1f} MB")
        logger.info(f"Available RAM: {mem_info_final.available / (1024**3):.1f} GB / {mem_info_final.total / (1024**3):.1f} GB")

print("\n" + "="*80)
print("STREAMING HDF5 DATASET PREPARATION COMPLETE (OPTIMIZED)")
print("=" * 80)

2025-10-26 16:28:43 - INFO - Available RAM: 26.9 GB / 48.0 GB
2025-10-26 16:28:43 - INFO - Initialized: /Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/testing -> /Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/hdf5_datasets/hdf5/streaming
2025-10-26 16:28:43 - INFO - Found 73 NPZ files (filtered 33 hidden files)
2025-10-26 16:28:43 - INFO - Modality types: ['TRIMMM', 'TRIMMM_with_tissue_mask']
2025-10-26 16:28:43 - INFO - Validating files in batches...
Validating batch 1/1: 100%|██████████| 73/73 [00:32<00:00,  2.26it/s]
2025-10-26 16:29:16 - INFO - Successfully validated 71 samples
2025-10-26 16:29:16 - INFO - Creating streaming/train: 49 files
2025-10-26 16:29:16 - INFO - Pass 1/2: Collecting statistics from 10 samples...
Sampling: 100%|██████████| 10/10 [01:09<00:00,  6.99s/it]
2025-10-26 16:30:25 - INFO - Pass 2/2: Writing data to HDF5...
Writing train: 100%|██████████| 49/49 [05:06<00:00,  6.25s/it]
2025-10-26 16:35:32 - INFO - Created /Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/hdf5_datasets/


HDF5 DATASET CREATION SUMMARY — STREAMING (OPTIMIZED)
       train:     49  train.h5 (8.84 GB)
  validation:     11  validation.h5 (1.82 GB)
        test:     11  test.h5 (1.82 GB)
       Total:     71
Output dir: /Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/hdf5_datasets/hdf5/streaming

VERIFY HDF5 — STREAMING (OPTIMIZED)

train.h5
  Samples: 49
  Split:   train
  Chans:   16
  Input μ/σ: 2652220162048.0000/inf
  Compression: gzip level 4
  Example: input (1078, 1278, 16), target (1078, 1278)
  Sample: mm_results_D15E_S6B_3_TRIMMM
  Modality: TRIMMM
  File size: 8.84 GB

._train.h5
  ERROR: Unable to synchronously open file (file signature not found)

validation.h5
  Samples: 11
  Split:   validation
  Chans:   16
  Input μ/σ: 11658915840.0000/352772406902784.0000
  Compression: gzip level 4
  Example: input (1078, 1278, 16), target (1078, 1278)
  Sample: mm_results_day15E_S8B_4_TRIMMM
  Modality: TRIMMM
  File size: 1.82 GB

._validation.h5
  ERROR: Unable to synchronously open file (f